# 13: AI Capstone Exercise — NWIS Data Analysis

In this capstone exercise you will complete a realistic hydrology data analysis workflow from start to finish: downloading USGS streamflow data, processing it with Pandas, and creating a publication-quality plot. The twist: for each step, you decide whether to write the code manually, use an AI assistant, or use AI with your own edits.

**Learning objectives:**
- Apply Python and AI skills from the entire Part 0 curriculum to a realistic task
- Practice choosing between manual coding and AI assistance for different sub-tasks
- Annotate your workflow to document *how* each section was produced
- Reflect on when AI helped, when it didn't, and what you learned

**Time estimate:** ~30 minutes

> **Note:** All AI-generated code examples in this notebook were captured ahead of time. No live AI API calls are made. If you have access to an AI coding assistant, you are encouraged to use it for the exercise sections. If not, you can work through the pre-captured examples and focus on evaluation and annotation.

---

## How to Use This Notebook

This exercise has **four steps**, each in its own section. For every step you will:

1. **Read the task description** — what needs to be accomplished
2. **Choose your approach** — write code yourself, use AI, or use AI and then edit
3. **Write or paste your code** in the provided cell
4. **Annotate your approach** using the label cell that follows each code cell

### Annotation Labels

After each code cell, mark your approach with **one** of these labels:

| Label | Meaning |
|-------|---------|
| **Manual** | You wrote the code entirely by hand |
| **AI-generated** | An AI assistant wrote the code and you used it as-is |
| **AI-assisted with edits** | An AI assistant wrote a draft and you modified it |

There are no wrong choices here. The goal is honest reflection on your workflow.

---

## Setup

Import the libraries we will use throughout the exercise.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dataretrieval import nwis

output_path = Path("./data/capstone")
output_path.mkdir(exist_ok=True, parents=True)

---

## Step 1: Download NWIS Streamflow Data

### The Task

Download **daily mean streamflow** data from USGS NWIS for the following site and time period:

| Parameter | Value |
|-----------|-------|
| Site number | `05407000` (Wisconsin River at Muscoda, WI) |
| Service | Daily values (`dv`) |
| Start date | `2018-10-01` (start of water year 2019) |
| End date | `2023-09-30` (end of water year 2023) |
| Parameter code | `00060` (discharge in cfs) |

Use the `dataretrieval` package (already imported as `nwis`) to fetch the data. Store the result in a variable called `daily`.

**Hint:** The function `nwis.get_record()` accepts `sites`, `service`, `start`, `end`, and `parameterCd` arguments.

### Your Code

In [ ]:
# Step 1: Download daily streamflow data
# YOUR CODE HERE

site = "05407000"
start = "2018-10-01"
end = "2023-09-30"

daily = nwis.get_record(
    sites=site,
    service="dv",
    start=start,
    end=end,
    parameterCd="00060",
)

print(f"Downloaded {len(daily)} daily records")
print(f"Date range: {daily.index.min()} to {daily.index.max()}")
daily.head()

### ✏️ Annotation — Step 1

**Approach used:** `________` *(Manual / AI-generated / AI-assisted with edits)*

**If you used AI, what prompt did you write?**

> *(paste your prompt here)*

**Why did you choose this approach for this step?**

> *(your reasoning)*

---

## Step 2: Process the Data with Pandas

### The Task

Starting from the `daily` DataFrame you downloaded in Step 1, perform the following processing:

1. **Extract the discharge column** — select the `00060_Mean` column (daily mean discharge in cfs)
2. **Convert units** — create a new column `flow_cms` that converts discharge from cubic feet per second (cfs) to cubic meters per second (cms) using the factor **0.0283168**
3. **Add a water year column** — create a column `water_year` where the water year starts on October 1 (e.g., October 2018 through September 2019 is water year 2019)
4. **Compute annual statistics** — create a summary DataFrame called `annual_stats` with one row per water year containing:
   - `mean_cms`: mean daily flow for that water year
   - `max_cms`: maximum daily flow
   - `min_cms`: minimum daily flow

**Ground truth check:** The mean daily flow for water year 2019 should be approximately **310–380 cms** (roughly 11,000–13,500 cfs). If your value is far outside this range, check your unit conversion.

### Your Code

In [ ]:
# Step 2: Process the data
# YOUR CODE HERE

# 2a. Convert units
daily["flow_cms"] = daily["00060_Mean"] * 0.0283168

# 2b. Add water year column
# Water year starts Oct 1: if month >= 10, water year = year + 1
daily["water_year"] = daily.index.year
daily.loc[daily.index.month >= 10, "water_year"] = (
    daily.loc[daily.index.month >= 10].index.year + 1
)

# 2c. Compute annual statistics
annual_stats = daily.groupby("water_year")["flow_cms"].agg(
    mean_cms="mean",
    max_cms="max",
    min_cms="min",
)

print("Annual statistics (cms):")
annual_stats

### ✅ Verification Checklist — Step 2

- [ ] Code runs without errors
- [ ] `flow_cms` values are roughly 0.028× the original cfs values
- [ ] Water year 2019 mean is approximately 310–380 cms
- [ ] Each water year has ~365 days of data
- [ ] All flow values are positive (physically reasonable)

In [ ]:
# Quick sanity checks
print("Records per water year:")
print(daily.groupby("water_year").size())
print()
print(f"All flow values positive: {(daily['flow_cms'] > 0).all()}")
print(f"WY2019 mean flow: {annual_stats.loc[2019, 'mean_cms']:.1f} cms")

### ✏️ Annotation — Step 2

**Approach used:** `________` *(Manual / AI-generated / AI-assisted with edits)*

**If you used AI, what prompt did you write?**

> *(paste your prompt here)*

**If you edited AI output, what did you change and why?**

> *(describe your edits)*

**Why did you choose this approach for this step?**

> *(your reasoning)*

---

## Step 3: Create a Publication-Quality Plot

### The Task

Create a **two-panel figure** suitable for a USGS report or journal publication:

**Panel A (top):** Daily hydrograph
- Plot daily mean streamflow (in cms) as a time series
- Use a blue line
- Add a horizontal dashed red line at the 5-year mean flow
- Label axes: x = "Date", y = "Streamflow (m³/s)"
- Add a legend

**Panel B (bottom):** Annual summary bar chart
- Bar chart of mean annual flow (cms) by water year
- Add error bars showing the range (min to max) for each year
- Label axes: x = "Water Year", y = "Mean Streamflow (m³/s)"

**Figure requirements:**
- Figure size: 10 × 8 inches
- Use `layout="constrained"` for clean spacing
- Add panel labels "(a)" and "(b)" in the upper-left corner of each panel
- Add a figure title: "Wisconsin River at Muscoda, WI (USGS 05407000)"
- Save the figure to `data/capstone/wisconsin_river_analysis.png` at 300 DPI

### Your Code

In [ ]:
# Step 3: Create publication-quality plot
# YOUR CODE HERE

fig, (ax1, ax2) = plt.subplots(
    nrows=2, ncols=1, figsize=(10, 8), layout="constrained"
)

# --- Panel A: Daily hydrograph ---
overall_mean = daily["flow_cms"].mean()

ax1.plot(daily.index, daily["flow_cms"], color="steelblue", linewidth=0.5,
         label="Daily mean flow")
ax1.axhline(overall_mean, color="red", linestyle="--", linewidth=1,
            label=f"5-year mean ({overall_mean:.0f} m³/s)")
ax1.set_ylabel("Streamflow (m³/s)")
ax1.set_xlabel("Date")
ax1.legend(loc="upper right")
ax1.text(0.02, 0.95, "(a)", transform=ax1.transAxes,
         fontsize=12, fontweight="bold", va="top")

# --- Panel B: Annual bar chart ---
water_years = annual_stats.index.values
means = annual_stats["mean_cms"].values
mins = annual_stats["min_cms"].values
maxs = annual_stats["max_cms"].values

# Error bars: distance from mean to min (lower) and mean to max (upper)
yerr_lower = means - mins
yerr_upper = maxs - means

ax2.bar(water_years, means, color="steelblue", alpha=0.7,
        yerr=[yerr_lower, yerr_upper], capsize=4,
        error_kw={"elinewidth": 1, "capthick": 1},
        label="Mean annual flow")
ax2.set_ylabel("Mean Streamflow (m³/s)")
ax2.set_xlabel("Water Year")
ax2.set_xticks(water_years)
ax2.legend(loc="upper right")
ax2.text(0.02, 0.95, "(b)", transform=ax2.transAxes,
         fontsize=12, fontweight="bold", va="top")

fig.suptitle("Wisconsin River at Muscoda, WI (USGS 05407000)",
             fontsize=14, fontweight="bold")

fig.savefig(output_path / "wisconsin_river_analysis.png", dpi=300)
print(f"Figure saved to {output_path / 'wisconsin_river_analysis.png'}")
plt.show()

### ✅ Verification Checklist — Step 3

- [ ] Figure has two panels stacked vertically
- [ ] Panel (a) shows a daily time series with a mean line
- [ ] Panel (b) shows a bar chart with error bars for each water year
- [ ] All axes are labeled with correct units (m³/s)
- [ ] Panel labels "(a)" and "(b)" are visible
- [ ] Figure title includes the site name and USGS site number
- [ ] Figure is saved as a PNG at 300 DPI

### ✏️ Annotation — Step 3

**Approach used:** `________` *(Manual / AI-generated / AI-assisted with edits)*

**If you used AI, what prompt did you write?**

> *(paste your prompt here)*

**If you edited AI output, what did you change and why?**

> *(describe your edits)*

**Why did you choose this approach for this step?**

> *(your reasoning)*

---

## Step 4: Summarize and Interpret

### The Task

Write a brief summary of the streamflow record. In the markdown cell below, answer these questions using the data and plot you produced:

1. Which water year had the **highest** mean annual flow? Which had the **lowest**?
2. What is the approximate **range** of daily flows over the 5-year period?
3. Are there any obvious **seasonal patterns** visible in the hydrograph?

Use the code cell to compute any additional statistics you need.

In [ ]:
# Step 4: Compute any additional statistics to support your summary
# YOUR CODE HERE

print("Annual statistics summary:")
print(annual_stats.to_string())
print()
print(f"Highest mean annual flow: WY {annual_stats['mean_cms'].idxmax()} "
      f"({annual_stats['mean_cms'].max():.1f} cms)")
print(f"Lowest mean annual flow:  WY {annual_stats['mean_cms'].idxmin()} "
      f"({annual_stats['mean_cms'].min():.1f} cms)")
print(f"Overall daily flow range: {daily['flow_cms'].min():.1f} – "
      f"{daily['flow_cms'].max():.1f} cms")

### Your Summary

*(Write your interpretation here. Replace the placeholder text.)*

1. **Highest/lowest mean annual flow:** ...
2. **Range of daily flows:** ...
3. **Seasonal patterns:** ...

### ✏️ Annotation — Step 4

**Approach used:** `________` *(Manual / AI-generated / AI-assisted with edits)*

**Why did you choose this approach for this step?**

> *(your reasoning)*

---

## Reflection Questions

Take a few minutes to reflect on your experience completing this exercise. Answer the questions below in the markdown cells provided.

### Question 1: Approach Choices

Look back at your annotations for Steps 1–4. Did you use the same approach for every step, or did you vary your approach? Why?

*(Your answer here)*

### Question 2: Where AI Helped Most

For which step (if any) was AI assistance most valuable? What made that step a good fit for AI?

*(Your answer here)*

### Question 3: Where Manual Coding Was Better

For which step (if any) did you prefer writing code manually? What made AI less useful for that step?

*(Your answer here)*

### Question 4: Verification

How did you verify that your results were correct? Did you rely on the ground truth values provided, visual inspection of the plot, or something else? Would you trust AI-generated code for this analysis without verification?

*(Your answer here)*

### Question 5: Professional Application

Imagine you need to produce a similar analysis for a USGS report — downloading data for 20 sites, processing each one, and generating standardized plots. How would you divide the work between manual coding and AI assistance? What parts would you want to write yourself, and what parts would you delegate to AI?

*(Your answer here)*

---

## Summary

In this capstone exercise you completed a full hydrology data analysis workflow:

1. **Downloaded** NWIS streamflow data using `dataretrieval`
2. **Processed** the data with Pandas (unit conversion, water year assignment, annual statistics)
3. **Visualized** the results with a publication-quality matplotlib figure
4. **Interpreted** the streamflow record

Along the way, you practiced the core skill this curriculum teaches: **making deliberate choices** about when to write code manually, when to use AI assistance, and when to combine both approaches. There is no single right answer — the best approach depends on the task, your familiarity with the tools, and the level of verification required.

### Key Takeaways

- **AI is strongest** at boilerplate and formatting tasks (plot styling, data I/O syntax)
- **Manual coding is strongest** when domain knowledge matters (unit conversions, water year logic, interpreting results)
- **Verification is always required** regardless of how the code was produced
- **Documenting your approach** (manual vs. AI) supports reproducibility and transparency in scientific work